# nb_03_check_my_access - "who am I and what can I see?"
**SYNTHETIC DATA · for training only · no real customer data**

Used live in **Demo 4**: run once as **dana** (Contributor) and once as **olivia** (Viewer + OneLake role `ReaderOn`).

* dana: sees all 13 provinces and the `sin` column. Contributors bypass OneLake security roles (lead finding).
* olivia: sees Ontario only and no `sin`/`date_of_birth`, **or** gets a friendly "access denied" if this Spark path cannot enforce row/column security. Either way it fails closed.

olivia is only a **Viewer** in `FGD-Security-Demo`, and Viewers can't run notebooks there. Run her copy from the optional sandbox workspace `FGD-Analyst-Sandbox` (created by `02-Deploy-FgdWorkspace.ps1 -CreateAnalystSandbox`). `workspace_name` points back at the demo workspace.

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data
# nb_03_check_my_access - parameters
workspace_name = "FGD-Security-Demo"   # where lh_silver lives (explicit, so it also works from a sandbox workspace)
silver_name    = "lh_silver"
table_name     = "customer"

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data
# Who am I? (prints identity claims only - never prints a token)
import base64, json
import notebookutils

def whoami():
    info = {}
    try:
        ctx = notebookutils.runtime.context            # VERIFY: key names can vary by runtime version
        for k in ("userName", "userId", "currentNotebookName", "currentWorkspaceName"):
            if k in ctx:
                info[k] = ctx[k]
    except Exception as e:
        info["runtime.context"] = f"unavailable ({type(e).__name__})"
    try:
        tok = notebookutils.credentials.getToken("pbi")
        payload = tok.split(".")[1]
        payload += "=" * (-len(payload) % 4)
        claims = json.loads(base64.urlsafe_b64decode(payload))
        info["token.upn"] = claims.get("upn") or claims.get("unique_name") or claims.get("preferred_username")
        info["token.oid"] = claims.get("oid")
    except Exception as e:
        info["token"] = f"unavailable ({type(e).__name__})"
    return info

me = whoami()
for k, v in me.items():
    print(f"{k:<22}: {v}")

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data

# Resolve workspace and item ids with semantic-link (sempy) so this notebook does
# NOT depend on a default lakehouse. All paths below are absolute abfss:// URLs.
import sempy.fabric as fabric
import notebookutils

ONELAKE = "onelake.dfs.fabric.microsoft.com"

def resolve_ws(name):
    """Workspace id for a name; '' means the workspace this notebook runs in."""
    if not name:
        return fabric.get_workspace_id()
    try:
        return fabric.resolve_workspace_id(name)
    except AttributeError:  # older semantic-link versions
        df = fabric.list_workspaces()
        return df[df["Name"] == name]["Id"].iloc[0]

def resolve_item(name, item_type, ws_id):
    """Item id for a display name + type (Lakehouse, SQLDatabase, ...)."""
    try:
        return fabric.resolve_item_id(name, type=item_type, workspace=ws_id)
    except Exception:
        df = fabric.list_items(type=item_type, workspace=ws_id)
        hit = df[df["Display Name"] == name]
        if hit.empty:
            raise ValueError(f"{item_type} '{name}' not found in workspace {ws_id}")
        return hit["Id"].iloc[0]

def item_root(ws_id, item_id):
    return f"abfss://{ws_id}@{ONELAKE}/{item_id}"

def friendly(e):
    msg = str(e)
    if any(s in msg for s in ("403", "Forbidden", "AuthorizationPermissionMismatch", "AccessDenied", "Unauthorized")):
        return ("ACCESS DENIED - OneLake security did not grant you this table through this engine. "
                "Expected for a Viewer without a OneLake role, or when the role has row/column rules that this "
                "path cannot enforce (it fails closed). Try the lh_silver SQL analytics endpoint instead.")
    if any(s in msg for s in ("404", "PathNotFound", "not found", "NotFound")):
        return "NOT FOUND - the item/table doesn't exist or you can't see it (Fabric hides what you can't access)."
    return "ERROR - " + msg[:400]

try:
    ws_id = resolve_ws(workspace_name)
    silver_id = resolve_item(silver_name, "Lakehouse", ws_id)
    table_path = f"{item_root(ws_id, silver_id)}/Tables/{table_name}"
    print(f"[step] target: {workspace_name}/{silver_name}/Tables/{table_name}")
except Exception as e:
    table_path = None
    print("[step] cannot resolve the lakehouse:", friendly(e))

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data
# What can I see?
from pyspark.sql import functions as F

if table_path:
    try:
        df = spark.read.format("delta").load(table_path)
        cols = df.columns
        print(f"[result] columns visible : {len(cols)} -> {cols}")
        print(f"[result] sin visible     : {'sin' in cols}")
        print(f"[result] dob visible     : {'date_of_birth' in cols}")
        by_prov = df.groupBy("province").count().orderBy(F.desc("count"))
        rows = by_prov.collect()
        print(f"[result] provinces visible: {len(rows)}  total rows: {sum(r['count'] for r in rows):,}")
        display(by_prov)
    except Exception as e:
        print("[result]", friendly(e))

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data
# Can I write? (Viewers should fail; Contributors succeed and the probe file is removed again)
if table_path:
    probe = table_path.split("/Tables/")[0] + "/Files/_access_probe.txt"
    try:
        notebookutils.fs.put(probe, "probe", True)
        notebookutils.fs.rm(probe)
        print("[result] write: ALLOWED (you can change data in lh_silver)")
    except Exception as e:
        print("[result] write: DENIED -", friendly(e)[:160])

## Pattern only - secrets from Azure Key Vault (shown, not run)
Never put secrets in notebooks or pipelines. Fetch them at run time with the caller's identity. Fabric redacts secret values in cell output. For unattended runs, prefer a workspace identity or a Fabric connection over secrets.

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data
# PATTERN ONLY - not executed in the demo (this kit deploys no Key Vault).
# key_vault_uri = "https://<your-key-vault>.vault.azure.net/"
# api_key = notebookutils.credentials.getSecret(key_vault_uri, "<secret-name>")   # caller needs Key Vault 'get' rights
# print(api_key)   # -> output shows [REDACTED]